# Model B: MobileNetV2 trained on SRGAN images
Train a fresh classifier using the same architecture and training settings as Model A. The only change is that Model B's training and validation inputs come from the validated SRGAN-generated training manifest.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import ImagePathDataset, load_splits
from applied_ai_midterm.generation import load_generated_training_manifest
from applied_ai_midterm.models import build_mobilenet_v2_classifier
from applied_ai_midterm.training import (
    create_train_validation_frames,
    fit_classifier,
    seed_everything,
    select_device,
)
from applied_ai_midterm.transforms import classifier_transform

In [ ]:
config = load_config(Path("configs/config.yaml"))
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
generated_frame = load_generated_training_manifest(
    Path("data/generated/manifest.csv"), splits.train
)
train_frame, validation_frame = create_train_validation_frames(
    generated_frame, validation_ratio=0.20, random_seed=config.random_seed
)
assert len(train_frame) + len(validation_frame) == len(splits.train)
display(pd.DataFrame({"subset": ["generated train", "generated validation"], "count": [len(train_frame), len(validation_frame)]}))

In [ ]:
# These transformations, loader settings, and seed match Model A exactly.
train_dataset = ImagePathDataset(
    train_frame, classifier_transform(training=True, image_size=config.high_resolution_size)
)
validation_dataset = ImagePathDataset(
    validation_frame, classifier_transform(training=False, image_size=config.high_resolution_size)
)
loader_kwargs = {"batch_size": config.classifier_batch_size, "num_workers": config.num_workers}
loader_generator = torch.Generator().manual_seed(config.random_seed)
train_loader = DataLoader(train_dataset, shuffle=True, generator=loader_generator, **loader_kwargs)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_kwargs)

In [ ]:
# Fresh MobileNetV2, same frozen features, Adam learning rate, epochs, and BCEWithLogitsLoss workflow as Model A.
model_b = build_mobilenet_v2_classifier(freeze_features=True).to(device)
optimizer = torch.optim.Adam(
    (parameter for parameter in model_b.parameters() if parameter.requires_grad), lr=1e-3
)
history = fit_classifier(
    model_b,
    train_loader,
    validation_loader,
    optimizer,
    Path("checkpoints/model_b_best.pt"),
    epochs=config.classifier_epochs,
    device=device,
    config={**asdict(config), "model": "Model B", "training_source": "SRGAN manifest"},
)

In [ ]:
history_frame = pd.DataFrame(
    {
        "epoch": [row["epoch"] for row in history],
        "training_loss": [row["training_loss"] for row in history],
        "validation_loss": [row["validation_loss"] for row in history],
    }
)
history_frame.plot(x="epoch", y=["training_loss", "validation_loss"], figsize=(8, 5))
plt.title("Model B loss")
plt.ylabel("BCEWithLogitsLoss")
plt.show()